# Proyecto de agronomia exploracion, analisis,

In [1]:
from __future__ import annotations

import argparse
import csv
from pathlib import Path

import matplotlib.pyplot as plt

from matplotlib.ticker import FuncFormatter
import pandas as pd
import requests


In [2]:
BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
FIG_DIR = BASE_DIR / "figuras"

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

ANIO_INICIAL = 2003
ANIO_FINAL = 2025

ANIOS = list(range(ANIO_INICIAL, ANIO_FINAL + 1))

URL_TEMPLATE = (
    "https://nube.agricultura.gob.mx/index.php"
    "?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO={anio}"
)

TOP_N = 15

print(f"Periodo de análisis: {ANIO_INICIAL}-{ANIO_FINAL}")
print(f"Número de años: {len(ANIOS)}")
print("Directorio raw:", RAW_DIR)

Periodo de análisis: 2003-2025
Número de años: 23
Directorio raw: b:\MCD2\JW\agro\proyecto_agronomia\data\raw


In [3]:
def descargar_datos_agricolas(anios, directorio):
    archivos = []

    for anio in anios:
        url = URL_TEMPLATE.format(anio=anio)
        ruta = directorio / f"agricola_{anio}.csv"

        if ruta.exists():
            print(f"[{anio}] Ya existe")
            archivos.append(ruta)
            continue

        print(f"[{anio}] Descargando...")

        response = requests.get(url, timeout=120)
        response.raise_for_status()

        ruta.write_bytes(response.content)

        print(
            f"[{anio}] Descargado "
            f"({ruta.stat().st_size / (1024**2):.2f} MB)"
        )

        archivos.append(ruta)

    return archivos


archivos_agricolas = descargar_datos_agricolas(
    ANIOS,
    RAW_DIR
)

[2003] Ya existe
[2004] Ya existe
[2005] Ya existe
[2006] Ya existe
[2007] Ya existe
[2008] Ya existe
[2009] Ya existe
[2010] Ya existe
[2011] Ya existe
[2012] Ya existe
[2013] Ya existe
[2014] Ya existe
[2015] Ya existe
[2016] Ya existe
[2017] Ya existe
[2018] Ya existe
[2019] Ya existe
[2020] Ya existe
[2021] Ya existe
[2022] Ya existe
[2023] Ya existe
[2024] Ya existe
[2025] Ya existe


In [4]:
print("Archivos encontrados:", len(archivos_agricolas))

for archivo in archivos_agricolas:
    print(archivo.name)

Archivos encontrados: 23
agricola_2003.csv
agricola_2004.csv
agricola_2005.csv
agricola_2006.csv
agricola_2007.csv
agricola_2008.csv
agricola_2009.csv
agricola_2010.csv
agricola_2011.csv
agricola_2012.csv
agricola_2013.csv
agricola_2014.csv
agricola_2015.csv
agricola_2016.csv
agricola_2017.csv
agricola_2018.csv
agricola_2019.csv
agricola_2020.csv
agricola_2021.csv
agricola_2022.csv
agricola_2023.csv
agricola_2024.csv
agricola_2025.csv


In [5]:
def leer_csv_agricola(ruta):
    codificaciones = ["utf-8", "latin-1", "cp1252"]

    ultimo_error = None

    for encoding in codificaciones:
        try:
            df = pd.read_csv(
                ruta,
                encoding=encoding,
                low_memory=False
            )

            return df, encoding

        except UnicodeDecodeError as error:
            ultimo_error = error

    raise ultimo_error

In [6]:
resumen_archivos = []

for archivo in archivos_agricolas:

    df_anio, encoding_usado = leer_csv_agricola(archivo)

    anio = int(archivo.stem.split("_")[1])

    resumen_archivos.append({
        "Anio": anio,
        "Filas": df_anio.shape[0],
        "Columnas": df_anio.shape[1],
        "Encoding": encoding_usado,
        "Duplicados": df_anio.duplicated().sum(),
        "Nulos": df_anio.isna().sum().sum(),
        "Estados": (
            df_anio["Nomestado"].nunique()
            if "Nomestado" in df_anio.columns else None
        ),
        "Municipios": (
            df_anio["Nommunicipio"].nunique()
            if "Nommunicipio" in df_anio.columns else None
        ),
        "Cultivos": (
            df_anio["Nomcultivo"].nunique()
            if "Nomcultivo" in df_anio.columns else None
        )
    })

resumen_anual = (
    pd.DataFrame(resumen_archivos)
      .sort_values("Anio")
      .reset_index(drop=True)
)

resumen_anual

,Anio,Filas,Columnas,Encoding,Duplicados,Nulos,Estados,Municipios,Cultivos
0,2003,35667,24,latin-1,0,0,32,2277,307.0
1,2004,36204,24,latin-1,0,0,32,2285,311.0
2,2005,36520,24,latin-1,0,0,32,2285,300.0
3,2006,36056,24,latin-1,0,0,32,2289,298.0
4,2007,36136,24,latin-1,0,0,32,2294,304.0
5,2008,35785,24,latin-1,0,0,32,2296,310.0
6,2009,35562,24,latin-1,0,0,32,2298,297.0
7,2010,36228,24,latin-1,0,0,32,2298,306.0
8,2011,36299,24,latin-1,0,0,32,2297,313.0
9,2012,36263,24,latin-1,0,0,32,2297,295.0


In [7]:
df_anio["Nomcultivo"].nunique() if "Nomcultivo" in df_anio.columns else None

288

In [8]:
for anio in range(2013, 2023):

    archivo = RAW_DIR / f"agricola_{anio}.csv"
    df_temp, encoding = leer_csv_agricola(archivo)

    print(f"\n===== {anio} =====")
    print(df_temp.columns.tolist())


===== 2013 =====
['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader', 'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo', 'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida', 'Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada', 'Cosechada', 'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']

===== 2014 =====
['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader', 'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo', 'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida', 'Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada', 'Cosechada', 'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']

===== 2015 =====
['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader', 'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo', 'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida', 'Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada', 'Cosechada', 'Siniestrada', 'V

Lo que cambió históricamente sobre los nombres de las columnas

Tenemos tres etapas:

Periodo	Cultivo	Precio
2003–2014	Nomcultivo	Precio
2015–2020	Nomcultivo Sin Um	Precio
2021–2025	Nomcultivo	Preciomediorural

Los NaN de 2015–2020 no eran datos faltantes. Nuestro código simplemente buscaba una columna que durante esos años tenía otro nombre.

In [9]:
for anio in [2014, 2015, 2019, 2020, 2021, 2022]:

    archivo = RAW_DIR / f"agricola_{anio}.csv"
    df_temp, _ = leer_csv_agricola(archivo)

    columna_cultivo = (
        "Nomcultivo Sin Um"
        if "Nomcultivo Sin Um" in df_temp.columns
        else "Nomcultivo"
    )

    print(f"\n===== {anio} =====")
    print("Columna:", columna_cultivo)
    print("Cultivos únicos:", df_temp[columna_cultivo].nunique())
    print(df_temp[columna_cultivo].dropna().unique()[:20])


===== 2014 =====
Columna: Nomcultivo
Cultivos únicos: 299
<StringArray>
[    'Avena forrajera en verde',    'Cebada forrajera en verde',
 'Triticale forrajero en verde',            'Pastos y praderas',
                      'Cebolla',                  'Chile verde',
                     'Cilantro',                       'Frijol',
                      'Lechuga',      'Maíz forrajero en verde',
                   'Maíz grano',                       'Pepino',
     'Sorgo forrajero en verde',       'Tomate rojo (jitomate)',
                'Alfalfa verde',                      'Durazno',
                    'Membrillo',                    'Nopalitos',
                         'Nuez',                        'Agave']
Length: 20, dtype: str

===== 2015 =====
Columna: Nomcultivo Sin Um
Cultivos únicos: 307
<StringArray>
[    'Avena forrajera en verde',    'Cebada forrajera en verde',
                      'Cebolla', 'Triticale forrajero en verde',
            'Pastos y praderas',            

In [10]:
def obtener_cultivos(anio):
    archivo = RAW_DIR / f"agricola_{anio}.csv"
    df_temp, _ = leer_csv_agricola(archivo)

    columna = (
        "Nomcultivo Sin Um"
        if "Nomcultivo Sin Um" in df_temp.columns
        else "Nomcultivo"
    )

    return set(
        df_temp[columna]
        .dropna()
        .str.strip()
        .unique()
    )

In [11]:
cultivos_2019 = obtener_cultivos(2019)
cultivos_2020 = obtener_cultivos(2020)
cultivos_2021 = obtener_cultivos(2021)
cultivos_2022 = obtener_cultivos(2022)

print("2019:", len(cultivos_2019))
print("2020:", len(cultivos_2020))
print("2021:", len(cultivos_2021))
print("2022:", len(cultivos_2022))

2019: 293
2020: 80
2021: 82
2022: 324


In [12]:
print(
    "Cultivos de 2019 ausentes en 2020:",
    len(cultivos_2019 - cultivos_2020)
)

print(
    "Cultivos de 2020 ausentes en 2021:",
    len(cultivos_2020 - cultivos_2021)
)

print(
    "Cultivos nuevos en 2021 respecto a 2020:",
    len(cultivos_2021 - cultivos_2020)
)

print(
    "Cultivos de 2022 ausentes en 2021:",
    len(cultivos_2022 - cultivos_2021)
)

Cultivos de 2019 ausentes en 2020: 213
Cultivos de 2020 ausentes en 2021: 10
Cultivos nuevos en 2021 respecto a 2020: 12
Cultivos de 2022 ausentes en 2021: 242


In [13]:
ausentes_2020_2021 = (
    (cultivos_2019 & cultivos_2022)
    - (cultivos_2020 | cultivos_2021)
)

print(
    "Presentes en 2019 y 2022, "
    "pero ausentes en 2020-2021:",
    len(ausentes_2020_2021)
)

sorted(ausentes_2020_2021)[:50]

Presentes en 2019 y 2022, pero ausentes en 2020-2021: 185


['Aceituna',
 'Acelga',
 'Achiote',
 'Albahaca',
 'Albricia',
 'Alcachofa',
 'Algarrobo',
 'Alhelí',
 'Anís',
 'Apio',
 'Arrayán',
 'Arvejón',
 'Arándano',
 'Ave del paraíso',
 'Baby back choi',
 'Bambú',
 'Bangaña',
 'Betabel',
 'Cacahuate',
 'Caimito',
 'Calabaza',
 'Calabaza semilla o chihua',
 'Camote',
 'Canola',
 'Canola forrajera',
 'Capulín',
 'Carambolo',
 'Caña de azúcar forrajera',
 'Caña de azúcar piloncillo',
 'Cebada forrajera en verde',
 'Centeno grano',
 'Cereza',
 'Chabacano',
 'Chay',
 'Chayote',
 'Chilacayote',
 'Chirimoya',
 'Chives',
 'Chícharo',
 'Cilantro',
 'Ciruela',
 'Clavel',
 'Clyptoria',
 'Coco fruta',
 'Col (repollo)',
 'Col de bruselas',
 'Colza',
 'Comino',
 'Coquia',
 'Cártamo forrajero en verde']

In [14]:
comunes_2020_2021 = cultivos_2020 & cultivos_2021
union_2020_2021 = cultivos_2020 | cultivos_2021

print("Comunes:", len(comunes_2020_2021))
print("Unión:", len(union_2020_2021))

jaccard = (
    len(comunes_2020_2021) /
    len(union_2020_2021)
)

print(f"Similitud Jaccard: {jaccard:.3f}")

Comunes: 70
Unión: 92
Similitud Jaccard: 0.761


Estamos calculando:

$$ J(C_{2020},C_{2021}) = \frac{|C_{2020}\cap C_{2021}|} {|C_{2020}\cup C_{2021}|} $$

Si resulta muy cercano a 1, sería especialmente interesante: significaría que los dos años anómalos no solo tienen ~80 cultivos, sino prácticamente los mismos ~80 cultivos.

### Anomalía de cobertura en 2020 y 2021

La corrección de los nombres históricos de las variables permitió
determinar que la ausencia inicial del número de cultivos entre 2015 y
2020 se debía a un cambio de nombre de la variable y no a valores
faltantes.

Sin embargo, después de considerar dicho cambio, se identificó una
discontinuidad importante en 2020 y 2021.

Mientras que los años cercanos contienen aproximadamente 300 cultivos
distintos, los cierres de 2020 y 2021 contienen únicamente 80 y 82,
respectivamente. Esta reducción coincide con una disminución del número
total de registros.

| Año | Registros | Cultivos únicos |
|---:|---:|---:|
| 2019 | 36,695 | 293 |
| 2020 | 28,661 | 80 |
| 2021 | 28,741 | 82 |
| 2022 | 37,555 | 324 |

Esta discontinuidad deberá investigarse antes de utilizar 2020 y 2021
como años directamente comparables con el resto de la serie histórica.
Con la evidencia disponible todavía no se atribuye una causa a esta
diferencia.

In [15]:
resumen_cultivos_estado = []

for anio in [2019, 2020, 2021, 2022]:

    archivo = RAW_DIR / f"agricola_{anio}.csv"
    df_temp, _ = leer_csv_agricola(archivo)

    columna_cultivo = (
        "Nomcultivo Sin Um"
        if "Nomcultivo Sin Um" in df_temp.columns
        else "Nomcultivo"
    )

    temp = (
        df_temp
        .groupby("Nomestado")[columna_cultivo]
        .nunique()
        .reset_index(name=str(anio))
    )

    resumen_cultivos_estado.append(temp)

In [16]:
cultivos_por_estado = resumen_cultivos_estado[0]

for temp in resumen_cultivos_estado[1:]:
    cultivos_por_estado = cultivos_por_estado.merge(
        temp,
        on="Nomestado",
        how="outer"
    )

cultivos_por_estado

,Nomestado,2019,2020,2021,2022
0,Aguascalientes,51,33,33,55
1,Baja California,84,42,41,103
2,Baja California Sur,73,37,37,80
3,Campeche,42,23,21,40
4,Chiapas,61,37,39,62
5,Chihuahua,62,32,34,64
6,Ciudad de México,65,30,31,65
7,Coahuila,44,27,26,46
8,Colima,51,32,32,50
9,Durango,68,43,41,67


In [17]:
cultivos_por_estado["Cambio_2019_2020"] = (
    cultivos_por_estado["2020"]
    - cultivos_por_estado["2019"]
)

cultivos_por_estado["Cambio_2021_2022"] = (
    cultivos_por_estado["2022"]
    - cultivos_por_estado["2021"]
)

cultivos_por_estado.sort_values(
    "Cambio_2019_2020"
)

,Nomestado,2019,2020,2021,2022,Cambio_2019_2020,Cambio_2021_2022
16,México,143,63,63,151,-80,88
20,Puebla,141,63,64,148,-78,84
14,Michoacán,116,56,54,118,-60,64
29,Veracruz,105,52,52,107,-53,55
13,Jalisco,109,59,59,109,-50,50
17,Nayarit,80,34,35,86,-46,51
10,Guanajuato,90,45,46,86,-45,40
11,Guerrero,92,47,48,92,-45,44
23,San Luis Potosí,91,47,49,93,-44,44
15,Morelos,89,46,47,89,-43,42


La discontinuidad de 2020–2021 afecta de manera generalizada la diversidad de cultivos registrada en las 32 entidades federativas.

In [18]:
cultivos_por_estado["Reduccion_pct_2020"] = (
    (
        cultivos_por_estado["2019"]
        - cultivos_por_estado["2020"]
    )
    / cultivos_por_estado["2019"]
    * 100
)

cultivos_por_estado["Recuperacion_pct_2022"] = (
    (
        cultivos_por_estado["2022"]
        - cultivos_por_estado["2021"]
    )
    / cultivos_por_estado["2021"]
    * 100
)

cultivos_por_estado[
    [
        "Nomestado",
        "2019",
        "2020",
        "Reduccion_pct_2020",
        "2021",
        "2022",
        "Recuperacion_pct_2022"
    ]
].sort_values(
    "Reduccion_pct_2020",
    ascending=False
)

,Nomestado,2019,2020,Reduccion_pct_2020,2021,2022,Recuperacion_pct_2022
17,Nayarit,80,34,57.500000,35,86,145.714286
16,México,143,63,55.944056,63,151,139.682540
20,Puebla,141,63,55.319149,64,148,131.250000
6,Ciudad de México,65,30,53.846154,31,65,109.677419
14,Michoacán,116,56,51.724138,54,118,118.518519
29,Veracruz,105,52,50.476190,52,107,105.769231
10,Guanajuato,90,45,50.000000,46,86,86.956522
1,Baja California,84,42,50.000000,41,103,151.219512
30,Yucatán,54,27,50.000000,27,52,92.592593
2,Baja California Sur,73,37,49.315068,37,80,116.216216


la reducción afectó a las 32 entidades, pero con distinta intensidad.

Nayarit pierde el 57.5% de sus cultivos registrados entre 2019 y 2020; Estado de México, 55.9%; Puebla, 55.3%; CDMX, 53.8%; y Michoacán, 51.7%. Incluso los estados menos afectados muestran reducciones considerables: Querétaro 21.2% y Nuevo León 21.4%.

Además, hay que interpretar con cuidado Recuperacion_pct_2022. Que Baja California marque 151.2% no significa que recuperó “151% de lo perdido”; significa que:

$$ \frac{103-41}{41}\times100=151.2\% $$

es decir, 2022 tuvo 151.2% más cultivos que 2021. Para comunicar “cuánto de la pérdida se recuperó”, necesitaríamos otra métrica.

¿La discontinuidad 2020–2021 afectó solamente a la diversidad de cultivos registrada o también a la magnitud de la producción agrícola?

In [19]:
resumen_cuantitativo = []

for archivo in archivos_agricolas:

    df_temp, _ = leer_csv_agricola(archivo)
    anio = int(archivo.stem.split("_")[1])

    resumen_cuantitativo.append({
        "Anio": anio,
        "Sembrada": df_temp["Sembrada"].sum(),
        "Cosechada": df_temp["Cosechada"].sum(),
        "Siniestrada": df_temp["Siniestrada"].sum(),
        "Valorproduccion": df_temp["Valorproduccion"].sum()
    })

resumen_cuantitativo = (
    pd.DataFrame(resumen_cuantitativo)
      .sort_values("Anio")
      .reset_index(drop=True)
)

resumen_cuantitativo

,Anio,Sembrada,Cosechada,Siniestrada,Valorproduccion
0,2003,21754408.89,20118772.75,1169115.14,1.924204e+11
1,2004,21874034.4,20191531.50,1267986.46,2.106400e+11
2,2005,21640071.79,18528490.86,2623897.02,2.002506e+11
3,2006,21436172.01,19967457.12,1014541.70,2.327090e+11
4,2007,21733229.76,20054633.61,1291854.10,2.699510e+11
5,2008,21902572.7,20502833.70,981157.36,3.059506e+11
6,2009,21832754.02,18688831.79,2750528.75,2.946619e+11
7,2010,21952745.02,20167772.84,1373118.91,3.317860e+11
8,2011,22136741.63,18093806.77,3467570.04,3.546569e+11
9,2012,21901600.26,20511050.61,905411.02,4.101603e+11


In [20]:
df_2021, _ = leer_csv_agricola(
    RAW_DIR / "agricola_2021.csv"
)

df_2021[
    [
        "Sembrada",
        "Cosechada",
        "Siniestrada",
        "Volumenproduccion",
        "Rendimiento",
        "Preciomediorural",
        "Valorproduccion"
    ]
].dtypes

Sembrada                 str
Cosechada            float64
Siniestrada          float64
Volumenproduccion    float64
Rendimiento          float64
Preciomediorural     float64
Valorproduccion      float64
dtype: object

In [21]:
sembrada_numerica = pd.to_numeric(
    df_2021["Sembrada"],
    errors="coerce"
)

problemas_sembrada = df_2021[
    sembrada_numerica.isna()
    & df_2021["Sembrada"].notna()
]

print("Valores problemáticos:", len(problemas_sembrada))

problemas_sembrada[
    [
        "Nomestado",
        "Nommunicipio",
        "Nomcultivo",
        "Sembrada"
    ]
].head(30)

Valores problemáticos: 2


,Nomestado,Nommunicipio,Nomcultivo,Sembrada
26789,Veracruz,Juan Rodríguez Clara,Piña,"10,271"
26799,Veracruz,José Azueta,Piña,"10,573"


In [22]:
problemas_sembrada["Sembrada"].unique()[:50]

<StringArray>
['10,271', '10,573']
Length: 2, dtype: str

In [23]:
df_2021["Sembrada"].map(type).value_counts()

Sembrada
<class 'str'>    28741
Name: count, dtype: int64

In [24]:
for anio in range(2003, 2026):

    archivo = RAW_DIR / f"agricola_{anio}.csv"
    df_temp, _ = leer_csv_agricola(archivo)

    print(
        anio,
        df_temp["Sembrada"].dtype
    )

2003 float64
2004 float64
2005 float64
2006 float64
2007 float64
2008 float64
2009 float64
2010 float64
2011 float64
2012 float64
2013 float64
2014 float64
2015 float64
2016 float64
2017 float64
2018 float64
2019 float64
2020 float64
2021 str
2022 float64
2023 float64
2024 float64
2025 float64


solo dos valores problematicos en total

In [25]:
df_2021.loc[
    df_2021["Sembrada"].isin(["10,271", "10,573"]),
    [
        "Nomestado",
        "Nommunicipio",
        "Nomcultivo",
        "Sembrada",
        "Cosechada",
        "Siniestrada",
        "Volumenproduccion",
        "Rendimiento",
        "Preciomediorural",
        "Valorproduccion"
    ]
]

,Nomestado,Nommunicipio,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion
26789,Veracruz,Juan Rodríguez Clara,Piña,"10,271",3413.0,0.0,158832.45,46.54,5000.00,794162250.0
26799,Veracruz,José Azueta,Piña,"10,573",3628.0,0.0,168596.82,46.47,5018.05,846027525.0


In [26]:
COLUMNAS_NUMERICAS = [
    "Sembrada",
    "Cosechada",
    "Siniestrada",
    "Volumenproduccion",
    "Rendimiento",
    "Precio",
    "Preciomediorural",
    "Valorproduccion"
]


def convertir_columnas_numericas(df):
    df = df.copy()

    for columna in COLUMNAS_NUMERICAS:

        if columna not in df.columns:
            continue

        df[columna] = (
            df[columna]
            .astype("string")
            .str.replace(",", "", regex=False)
        )

        df[columna] = pd.to_numeric(
            df[columna],
            errors="coerce"
        )

    return df

In [27]:
resumen_cuantitativo = []

for archivo in archivos_agricolas:

    df_temp, _ = leer_csv_agricola(archivo)
    df_temp = convertir_columnas_numericas(df_temp)

    anio = int(archivo.stem.split("_")[1])

    resumen_cuantitativo.append({
        "Anio": anio,
        "Sembrada": df_temp["Sembrada"].sum(),
        "Cosechada": df_temp["Cosechada"].sum(),
        "Siniestrada": df_temp["Siniestrada"].sum(),
        "Valorproduccion": df_temp["Valorproduccion"].sum()
    })

resumen_cuantitativo = (
    pd.DataFrame(resumen_cuantitativo)
    .sort_values("Anio")
    .reset_index(drop=True)
)

In [28]:
resumen_cuantitativo[
    resumen_cuantitativo["Anio"].between(2018, 2023)
]

,Anio,Sembrada,Cosechada,Siniestrada,Valorproduccion
15,2018,21163051.24,20270413.03,411301.12,6.414313e+11
16,2019,20664424.08,19361663.28,844098.54,6.786288e+11
17,2020,18125470.40,17048486.50,702929.19,6.309332e+11
18,2021,18151034.91,17229616.57,541695.91,6.928286e+11
19,2022,20567358.34,20011525.54,179893.70,8.848763e+11
20,2023,20023594.61,18383456.31,1177045.39,9.218770e+11


la desaparición de ~200 cultivos en 2020–2021 no produjo una caída proporcional en las magnitudes agrícolas nacionales.

In [29]:
for columna in [
    "Sembrada",
    "Cosechada",
    "Siniestrada",
    "Valorproduccion"
]:
    resumen_cuantitativo[f"{columna}_Cambio_pct"] = (
        resumen_cuantitativo[columna]
        .pct_change()
        .mul(100)
    )

resumen_cuantitativo[
    resumen_cuantitativo["Anio"].between(2018, 2023)
]

,Anio,Sembrada,Cosechada,Siniestrada,Valorproduccion,Sembrada_Cambio_pct,Cosechada_Cambio_pct,Siniestrada_Cambio_pct,Valorproduccion_Cambio_pct
15,2018,21163051.24,20270413.03,411301.12,6.414313e+11,-1.980139,-2.507200,23.327673,8.973408
16,2019,20664424.08,19361663.28,844098.54,6.786288e+11,-2.356121,-4.483134,105.226414,5.799151
17,2020,18125470.40,17048486.50,702929.19,6.309332e+11,-12.286593,-11.947201,-16.724274,-7.028238
18,2021,18151034.91,17229616.57,541695.91,6.928286e+11,0.141042,1.062441,-22.937343,9.810131
19,2022,20567358.34,20011525.54,179893.70,8.848763e+11,13.312318,16.146088,-66.790648,27.719368
20,2023,20023594.61,18383456.31,1177045.39,9.218770e+11,-2.643819,-8.135658,554.300506,4.181461


¿Qué pasó de 2019 a 2020?

Las variaciones calculadas son:

$$ \text{Sembrada}: -12.29\% $$ $$ \text{Cosechada}: -11.95\% $$ $$ \text{Siniestrada}: -16.72\% $$ $$ \text{Valorproduccion}: -7.03\% $$

Pero al mismo tiempo:

$$ 293\text{ cultivos}\rightarrow80 $$

que equivale aproximadamente a una reducción de:

$$ \frac{293-80}{293}\times100\approx72.7\% $$

lo interesante:

El número de cultivos distintos cae aproximadamente 73%, pero la superficie sembrada cae solamente 12.3%.